In [1]:
import sys
import os
from scipy.spatial import cKDTree

chemin_lib = os.path.abspath("..")

# Add path to Python path
sys.path.append(chemin_lib)

import ast

import pandas as pd
import wfdb

from sklearn.preprocessing import MultiLabelBinarizer
import torch
import numpy as np
import SimpleITK as sitk 

from metriclib.metric import StreamMetric, MetricResult, TabularMetric
from metriclib.report import Report

from metriclib.data import Dataset

In [2]:
### CHAOSDataset

base_dir = os.path.join("..", "sample-data", "CHAOS_dataset")
agg_path = os.path.join(base_dir, "CHAOS_metadatas.csv")
aggregation_df = pd.read_csv(agg_path)
    
class CHAOSDataset(Dataset):
    def __init__(self):
        df = pd.read_csv(os.path.join(base_dir, "CHAOS_metadatas.csv"))
        self.metadata = df
        self.metadata["CHAOS_ID"] = self.metadata["CHAOS_ID"].astype(int)
        self.labels = self.metadata["CHAOS_ID"]

    def __len__(self):
        return len(self.metadata.index)

    def __getitem__(self, index):
        metadata = self.metadata.iloc[index].to_dict()
        metadata["idx"] = int(index)
        num_id = str(self.metadata["CHAOS_ID"].iloc[index])
        record_path = os.path.join("..", "sample-data", "CHAOS_dataset", "img_NIFTI", "CHAOS_"+str(num_id)+".nii.gz")
        seg1_path = os.path.join("..", "sample-data", "CHAOS_dataset", "seg_TotalSegmentator_NIFTI", "CHAOS_"+str(num_id)+".nii.gz")
        seg1 = sitk.ReadImage(seg1_path)
        seg2_path = os.path.join("..", "sample-data", "CHAOS_dataset", "seg_nnInteractive_NIFTI", "CHAOS_"+str(num_id)+".nii.gz")
        seg2 = sitk.ReadImage(seg2_path)
        x = sitk.ReadImage(record_path)
        y = (seg1, seg2) # y : segmentation masks, tuple of the segmentations to compare

        return x, y, metadata

dataset_chaos = CHAOSDataset()

In [3]:
report = Report(datasets=[dataset_chaos])
report.add_metric("DICESimilarityCoefficient")
report.add_metric("IntersectionOverUnion")
report.add_metric("HausdorffDistance")
report.add_metric("HausdorffDistance95")
test = report.generate()

100%|██████████| 4/4 [00:15<00:00,  3.93s/it]


In [4]:
test

([{'metric': metriclib.metrics.measurement_process.DICESimilarityCoefficient,
   'reference': None,
   'metric_config': None,
   'dataset': 'CHAOSDataset',
   'name': None,
   'result': MetricResult(description='DICE Score between two segmentations', value=[0.9557618292959021, 0.954658851454453, 0.9496780735107732, 0.9502340455114242], cluster=None, threshold=0)},
  {'metric': metriclib.metrics.measurement_process.IntersectionOverUnion,
   'reference': None,
   'metric_config': None,
   'dataset': 'CHAOSDataset',
   'name': None,
   'result': MetricResult(description='Intersection over Union Score between two segmentations', value=[0.9152718758130257, 0.9132510021084825, 0.9041780901262695, 0.9051865717767142], cluster=None, threshold=0)},
  {'metric': metriclib.metrics.measurement_process.HausdorffDistance,
   'reference': None,
   'metric_config': None,
   'dataset': 'CHAOSDataset',
   'name': None,
   'result': MetricResult(description='maximum Hausdorff Distance between two segment